In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import DQN, PPO
from sklearn.preprocessing import MinMaxScaler
from fpdf import FPDF

# 1. CONFIGURACIÓN
MODE = "SP500"  # "BTC" o "SP500"
ALGO = "ppo"  # "dqn" o "ppo"

if MODE == "BTC":
    foldername, data_name = "BTC", "BTCUSDT"
    TRAIN_END_DATE, val_size = '2024-12-31', 4698
else:
    foldername, data_name = "SP500", "SPY"
    TRAIN_END_DATE, val_size = '2025-04-10', 535

FROM_SAFE, UNTIL_SAFE = '2021-12-31_00-00-00', '2025-07-31_00-00-00'
INITIAL_BALANCE, FEE = 1000, 0.001
MODELS = ['raw', 'baseline', 'temporal-ae', 'cpc', 'transformer', 'combinado']
SEEDS = [42, 123, 456, 789, 1011]

# Importar entorno
sys.path.append('../')
from src.SRLTradingEnv_v2 import SRLTradingEnv_v2

# Carpeta temporal para las gráficas que irán al PDF
TMP_IMG_DIR = "temp_plots"
os.makedirs(TMP_IMG_DIR, exist_ok=True)

# ==========================================================
# 2. FUNCIONES DE APOYO
# ==========================================================

def load_data_all_types(model_type, tf="1h"):
    price_path = f'../data/{foldername}/01-output-{data_name}_{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}-log-return.csv'
    df_prices_full = pd.read_csv(price_path, index_col='date', parse_dates=True)
    
    if model_type == 'baseline':
        path = f'../data/{foldername}/02-baseline-{data_name}_{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}-technical-indicators.csv'
        df_feat = pd.read_csv(path, index_col='date', parse_dates=True)
    elif model_type == 'combinado':
        paths = {
            'ae': f'../data/{foldername}/02-srl-temporal-ae-{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}.csv',
            'cpc': f'../data/{foldername}/02-srl-cpc-{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}.csv',
            'trans': f'../data/{foldername}/02-srl-transformer-{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}.csv'
        }
        df_feat = pd.read_csv(paths['ae'], index_col='date', parse_dates=True).join(
                  pd.read_csv(paths['cpc'], index_col='date', parse_dates=True), how='inner').join(
                  pd.read_csv(paths['trans'], index_col='date', parse_dates=True), how='inner')
    elif model_type == 'raw':
        df_feat = df_prices_full[['close']].copy().rename(columns={'close': 'raw_ret'}).shift(1)
    else:
        df_feat = pd.read_csv(f'../data/{foldername}/02-srl-{model_type}-{tf}-from-{FROM_SAFE}-until-{UNTIL_SAFE}.csv', index_col='date', parse_dates=True)
    
    df_feat = df_feat.drop(columns=['open', 'high', 'low', 'close', 'volume', 'tradecount'], errors='ignore')
    combined = df_feat.join(df_prices_full[['close']], how='inner').replace([np.inf, -np.inf], np.nan).ffill().bfill().dropna()
    return combined.drop(columns=['close']), combined['close']

# ==========================================================
# 3. PROCESAMIENTO
# ==========================================================
all_results = []
best_plot_paths = {}

print(f"🚀 Procesando {MODE} - {ALGO.upper()}...")

for m in MODELS:
    best_ret = -999
    for s in SEEDS:
        try:
            feat, price = load_data_all_types(m)
            scaler = MinMaxScaler()
            train_mask = feat.index <= TRAIN_END_DATE
            scaler.fit(feat.loc[train_mask])
            feat_scaled = pd.DataFrame(scaler.transform(feat), index=feat.index, columns=feat.columns)
            
            f_test, p_test = feat_scaled[feat_scaled.index > TRAIN_END_DATE], price[price.index > TRAIN_END_DATE]
            path = f"../RL_outputs/{foldername}/models/{ALGO}_{m}_seed_{s}.zip"
            agent = (DQN if ALGO == "dqn" else PPO).load(path)
            
            env = SRLTradingEnv_v2(f_test, p_test, initial_balance=INITIAL_BALANCE, fee=FEE)
            obs, _ = env.reset()
            done, nw_hist, current_pos = False, [], 1
            l_ops, s_ops, exits = 0, 0, 0
            l_idx, s_idx = [], []

            step = 0
            while not done:
                action, _ = agent.predict(obs, deterministic=True)
                if action != current_pos:
                    if action == 2: l_ops += 1; l_idx.append(step)
                    elif action == 0: s_ops += 1; s_idx.append(step)
                    elif action == 1: exits += 1
                    current_pos = action
                obs, _, done, _, info = env.step(action)
                nw_hist.append(info['net_worth'])
                step += 1

            f_ret = ((nw_hist[-1]/1000)-1)*100
            m_ret = ((p_test.iloc[-1]/p_test.iloc[0])-1)*100
            all_results.append([m.upper(), str(s), str(l_ops), str(s_ops), str(exits), f"{f_ret:.2f}%", f"{(f_ret-m_ret):.2f}%"])

            if f_ret > best_ret:
                best_ret = f_ret
                plt.figure(figsize=(10, 4))
                plt.plot(nw_hist, label="Equity Curve", color='blue')
                plt.axhline(y=1000, color='red', linestyle='--', alpha=0.3)
                plt.title(f"{m.upper()} Seed {s} (Retorno: {f_ret:.2f}%)")
                plt.grid(alpha=0.2)
                p_path = f"{TMP_IMG_DIR}/{m}.png"
                plt.savefig(p_path)
                plt.close()
                best_plot_paths[m] = p_path
        except: continue

# ==========================================================
# 4. GENERACIÓN DEL PDF (FPDF2)
# ==========================================================
class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 16)
        self.set_text_color(31, 97, 141)
        self.cell(0, 10, f"Informe Tecnico RL - {MODE} ({ALGO.upper()})", ln=True, align="C")
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font("Arial", "I", 8)
        self.cell(0, 10, f"Pagina {self.page_no()}", align="C")

pdf = PDF()
pdf.add_page()

# Tabla de Resultados
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 10, "1. Tabla Comparativa de Modelos", ln=True)
pdf.set_font("Arial", "B", 10)
header = ["Modelo", "Seed", "Long", "Short", "Exit", "Retorno", "Alpha"]
col_widths = [35, 20, 20, 20, 20, 30, 30]

for i, h in enumerate(header):
    pdf.cell(col_widths[i], 10, h, 1, align="C")
pdf.ln()

pdf.set_font("Arial", "", 9)
for row in all_results:
    for i, datum in enumerate(row):
        pdf.cell(col_widths[i], 8, datum, 1, align="C")
    pdf.ln()

# Añadir Gráficas
pdf.add_page()
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 10, "2. Curvas de Patrimonio (Mejores Semillas)", ln=True)

for m, img_path in best_plot_paths.items():
    if os.path.exists(img_path):
        pdf.set_font("Arial", "B", 10)
        pdf.cell(0, 10, f"Modelo: {m.upper()}", ln=True)
        pdf.image(img_path, w=170)
        pdf.ln(5)

pdf_output = f"Informe_Final_{MODE}_{ALGO}.pdf"
pdf.output(pdf_output)
print(f"✅ ¡Informe guardado como {pdf_output}!")

🚀 Procesando SP500 - PPO...


c:\Users\javil\OneDrive\Escritorio\Uma\25-26\1Cuatri\tfg\TFG-Trading-SRL\RTS_TDA_DRL-main\venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
c:\Users\javil\OneDrive\Escritorio\Uma\25-26\1Cuatri\tfg\TFG-Trading-SRL\RTS_TDA_DRL-main\venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). 

✅ ¡Informe guardado como Informe_Final_SP500_ppo.pdf!


C:\Users\javil\AppData\Local\Temp\ipykernel_23652\3060631092.py:125: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  self.set_font("Arial", "B", 16)
C:\Users\javil\AppData\Local\Temp\ipykernel_23652\3060631092.py:127: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, f"Informe Tecnico RL - {MODE} ({ALGO.upper()})", ln=True, align="C")
C:\Users\javil\AppData\Local\Temp\ipykernel_23652\3060631092.py:139: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 12)
C:\Users\javil\AppData\Local\Temp\ipykernel_23652\3060631092.py:140: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "1. Tabla Comparativa de Modelos", ln=True)